
## 🚀 Hire-Er-Key: Multi-Dimensional Candidate Matching System

### 🔹 Version 1.0 — Core System (Completed)

In Version 1.0, we successfully developed an end-to-end AI-powered candidate matching system for a **GenAI Engineer role**, implementing all core requirements of the assignment:

* Hybrid Retrieval (FAISS + BM25)
* Structured Skill Matching using weighted clusters
* Threshold-based filtering
* Comparative ranking using LLM reasoning
* Explainable outputs (strengths & gaps)
* Agent-based routing via **Hero() dispatcher**

This version demonstrates a fully functional **RAG-based hiring intelligence pipeline**, capable of evaluating and ranking candidates with explainability.

---

### 🚀 Version 2.0 — Planned Enhancements

To improve flexibility, realism, and usability, the following upgrades are proposed:

* **Dynamic Job Description Support**
  → Allow the system to adapt to *any role*, not just GenAI Engineer

* **Dynamic Thresholding**
  → Adjust selection criteria based on role type (internship, entry-level, senior)

* **Top-K Candidate Selection**
  → Return only the most relevant candidates instead of all filtered ones

* **Candidate Name Extraction**
  → Replace file-based identifiers with actual candidate names

* **Semantic Skill Equivalence**
  → Use LLM reasoning to match equivalent skills (e.g., “LLM orchestration” ≈ LangGraph)

---

### 🔥 Version 3.0 — Future Vision (Advanced Intelligence)

A major extension of this system includes:

* **Alternate Role Recommendation System**
  → If a candidate is not suitable for the given role, the system suggests **better-fitting alternative roles** based on their profile

This introduces a more human-centric hiring approach, ensuring candidates are not simply rejected but redirected toward opportunities where they can succeed.

---

### ⚡ Additional Improvements (V3 Direction)

* **Latency Reduction**
  → Optimize LLM usage via batch processing

* **Enhanced UI/UX**
  → Improve visual presentation and interactivity for a more product-like experience

---

### 🎯 Demo Strategy (Important Note)

Due to API rate limits and latency constraints in free-tier LLM usage, the UI operates in a **controlled demo mode**:

* Core pipeline (metadata extraction, scoring, ranking) is **precomputed in the backend**
* UI acts as a **presentation layer**, displaying already processed results
* If backend processing is not completed, appropriate error messages are shown

> This approach ensures a **stable, fast, and reliable demonstration**, while the system itself fully supports dynamic, real-time execution.

---

### 🧠 Summary

Hire-Er-Key is not just a matching tool, but a **multi-stage AI decision system** combining:

* Retrieval
* Structured evaluation
* LLM reasoning
* Explainability

It lays the foundation for a scalable, intelligent hiring assistant with strong potential for real-world application.

Hire-Er-Key V1.0(Testing+ Initial UI Phase)

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu rank-bm25 openai gradio

Right Now We are Doing Backend Testing, Before Heading Forward with UI

First Part of Testing-Ingestion of Files

In [2]:
from google.colab import files
uploaded = files.upload()

Saving resumes.zip to resumes.zip


Creating ResumeStockPile()- Our Pile of Resume's with relevant Info

In [3]:
from pypdf import PdfReader
import zipfile
import os

def ResumeStockPile(zip_path="resumes.zip"):
    resumes = []

    # unzip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("resumes_folder")

    # walk through ALL folders inside
    for root, dirs, files in os.walk("resumes_folder"):
        for file in files:
            if file.endswith(".pdf"):
                path = os.path.join(root, file)

                reader = PdfReader(path)
                text = ""

                for page in reader.pages:
                    text += page.extract_text() + "\n"

                resumes.append({
                    "filename": file,
                    "text": text
                })

    print(f"Loaded {len(resumes)} resumes")
    return resumes

V2  Minimal Addition!!! Extract Name

In [4]:
def ExtractName(resume_text):
    for line in resume_text.split("\n"):
        if "name" in line.lower():
            return line.replace("Name:", "").strip()
    return "Unknown"

Testing ResumeStockPile()

In [ ]:
resumes = ResumeStockPile()
print(resumes[0]["text"][:300])
candidate_names = [ExtractName(r["text"]) for r in resumes] #V2.0 Minimal Change

✅ Status Right Now

✔ Libraries installed
✔ ZIP upload working
✔ ResumeStockPile() working
✔ Text extraction working

Next Step--> ExtractMetaData()- here we will define the llm+ Extract Important Data like Skills,Experience,etc. from the Resume Pile we got.

Getting The LLM Ready First(We are gonna use it a lot)

In [6]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("StepUp")
)

ExtractMetaData() Function

In [7]:
import json

def ExtractMetadata(resume_text):
    prompt = f"""
    Extract structured information from the following resume.

    Return ONLY valid JSON with:
    - skills (list)
    - experience_years (number)
    - tools/frameworks (list)
    - projects (list)
    - implicit_signals (list)

    Resume:
    {resume_text}
    """

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except:
        print("JSON parsing failed, raw output:")
        print(content)
        return None

Testing ExtractMetaData() Function

In [9]:
metadata_list = []

for r in resumes:
    meta = ExtractMetadata(r["text"])
    metadata_list.append(meta)

print(metadata_list[1])

{'skills': ['Customer Support', 'CRM tools', 'Basic SQL', 'Documentation'], 'experience_years': 1, 'tools/frameworks': ['Google Sheets'], 'projects': ['Ticket Tracking Sheet Automation'], 'implicit_signals': ['Strong written communication', 'Interpersonal skills', 'Tech-savviness']}


Now Storing these Resume Metadata into Vector Database as Embeddings-BuildVectorStore() Function

In [10]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [11]:
def BuildVectorStore(resumes, metadata_list):

    model = SentenceTransformer("all-MiniLM-L6-v2")

    texts = [r["text"] for r in resumes]

    # create embeddings
    embeddings = model.encode(texts)

    embeddings = np.array(embeddings).astype("float32")

    # create FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    print(f"FAISS index built with {index.ntotal} vectors")

    return index, texts, metadata_list

In [12]:
index, texts, metadata_store = BuildVectorStore(resumes, metadata_list)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built with 6 vectors


GenAIJobEater()- Processing the JD Now(for V1.0, We have provided the JD as hardcode but V2.0 may have Better Improvements)

In [13]:
import json

def GenAIJobEater(jd_text):
    prompt = f"""
Analyze the following Job Description and extract structured skill clusters.

STRICT RULES:
- You MUST populate all four categories:
  must_have, important, nice_to_have, implicit
- If not explicitly present, infer reasonable items
- Do NOT leave any list empty

Return ONLY valid JSON with:
- must_have (list)
- important (list)
- nice_to_have (list)
- implicit (list)
- weights (object with keys: must_have, important, nice_to_have, implicit)

Ensure weights sum to 1.0

Job Description:
{jd_text}
"""

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except:
        print("JSON parsing failed, raw output:")
        print(content)
        return None

Providing the Gen AI Engineer JD(for V1.0, Hardcode)

In [14]:
jd_text = """
We are hiring a GenAI Engineer.

Requirements:
- Strong Python skills
- Experience with LLMs and RAG pipelines
- Knowledge of LangChain or similar frameworks
- Familiarity with vector databases (FAISS, Pinecone)

Preferred:
- Experience with deployment (Docker, APIs)
- Understanding of ML fundamentals
"""

Testing GenAIJobEater()

In [15]:
jd_struct = GenAIJobEater(jd_text)
print(jd_struct)

{'must_have': ['Python programming', 'LLM and RAG pipeline development', 'LangChain or similar frameworks', 'Vector databases (FAISS, Pinecone)'], 'important': ['Cloud infrastructure experience (AWS, GCP, Azure)', 'Machine learning frameworks (TensorFlow, PyTorch)', 'Data preprocessing and management', 'Prompt engineering techniques'], 'nice_to_have': ['Deployment skills (Docker, APIs)', 'Understanding of ML fundamentals'], 'implicit': ['Version control (Git)', 'Linux/Unix proficiency', 'Software development best practices', 'Team collaboration and communication'], 'weights': {'must_have': 0.4, 'important': 0.3, 'nice_to_have': 0.2, 'implicit': 0.1}}


HybridRetrieve()- BM25+Dense(using FAISS) and then we will combine to make a Hybrid Score

BM25 Setup

In [16]:
from rank_bm25 import BM25Okapi

# tokenize resumes
tokenized_corpus = [text.lower().split() for text in texts]

bm25 = BM25Okapi(tokenized_corpus)

HybridRetrieve() Function

In [36]:
def HybridRetrieve(jd_text, texts, index, top_k=6):

    # --- BM25 ---
    tokenized_query = jd_text.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)

    # normalize BM25
    bm25_scores = np.array(bm25_scores)
    bm25_scores = bm25_scores / bm25_scores.max()

    # --- Dense (FAISS) ---
    model = SentenceTransformer("all-MiniLM-L6-v2")

    query_embedding = model.encode([jd_text]).astype("float32")

    D, I = index.search(query_embedding, top_k)

    dense_scores = np.zeros(len(texts))

    for rank, idx in enumerate(I[0]):
        dense_scores[idx] = 1 / (1 + D[0][rank])  # convert distance → similarity

    # normalize dense
    if dense_scores.max() > 0:
        dense_scores = dense_scores / dense_scores.max()

    # --- Combine ---
    final_scores = 0.6 * bm25_scores + 0.4 * dense_scores

    # rank
    ranked_indices = np.argsort(final_scores)[::-1]

    results = []

    for rank, idx in enumerate(ranked_indices):
      results.append({
        "rank": rank + 1,
        "filename": resumes[idx]["filename"],   # 👈 ADD THIS
        "score": float(final_scores[idx])
    })
    return results

In [37]:
results = HybridRetrieve(jd_text, texts, index)

for r in results:
    print(f"Rank {r['rank']} | {r['filename']} | Score: {r['score']:.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rank 1 | Resume-ArjunMehta.pdf | Score: 1.000
Rank 2 | Resume-RiyaSharma.pdf | Score: 0.729
Rank 3 | Resume-NehaYadav.pdf | Score: 0.723
Rank 4 | Resume-KunalVerma.pdf | Score: 0.447
Rank 5 | Resume-SnehaKulKarni.pdf | Score: 0.379
Rank 6 | Resume-RahulSaini.pdf | Score: 0.342


Ok now we have RAW SELECTION, just based off resume and nothing else. There is Still a Problem, Rank 3 Getting a higher rank even if you see their skill, it is WAY DIFFERENT THAN  A GEN AI ENGINEER.

 So we have to build a SimpleSkillMatcher()- For simple means matching just based off skill in jd vs skill in resume. Simple. Adding Complexity once it works

In [38]:
def SimpleSkillMatcher(metadata_list, jd_struct):

    # combine ALL JD skills
    jd_skills = (
        jd_struct["must_have"] +
        jd_struct["important"] +
        jd_struct["nice_to_have"] +
        jd_struct["implicit"]
    )

    jd_skills = [s.lower() for s in jd_skills]

    results = []

    for i, meta in enumerate(metadata_list):

        candidate_skills = meta["skills"]
        candidate_skills = [s.lower() for s in candidate_skills]

        matched = 0

        for skill in jd_skills:
            for c_skill in candidate_skills:
                if skill in c_skill or c_skill in skill:
                    matched += 1
                    break

        total = len(jd_skills)
        score = matched / total if total > 0 else 0

        results.append({
            "candidate": resumes[i]["filename"],
            "matched": matched,
            "total": total,
            "score": score
        })

    return results

In [39]:
simple_scores = SimpleSkillMatcher(metadata_list, jd_struct)

for r in simple_scores:
    print(r)

{'candidate': 'Resume-RahulSaini.pdf', 'matched': 1, 'total': 14, 'score': 0.07142857142857142}
{'candidate': 'Resume-NehaYadav.pdf', 'matched': 0, 'total': 14, 'score': 0.0}
{'candidate': 'Resume-RiyaSharma.pdf', 'matched': 6, 'total': 14, 'score': 0.42857142857142855}
{'candidate': 'Resume-KunalVerma.pdf', 'matched': 1, 'total': 14, 'score': 0.07142857142857142}
{'candidate': 'Resume-SnehaKulKarni.pdf', 'matched': 2, 'total': 14, 'score': 0.14285714285714285}
{'candidate': 'Resume-ArjunMehta.pdf', 'matched': 6, 'total': 14, 'score': 0.42857142857142855}


Now Adding Scores to musthaves, important, nice to have and implicit and ranking based on that using SkillMatcher() (upgrade to SimpleSkillMatcher())

In [40]:
def SkillMatcher(metadata_list, jd_struct):

    weights = jd_struct["weights"]

    results = []

    for i, meta in enumerate(metadata_list):

        candidate_skills = [s.lower() for s in meta["skills"]]

        def compute_match(jd_list):
            jd_list = [s.lower() for s in jd_list]

            matched = 0
            for skill in jd_list:
                for c_skill in candidate_skills:
                    if skill in c_skill or c_skill in skill:
                        matched += 1
                        break

            total = len(jd_list)
            return matched / total if total > 0 else 0

        must_score = compute_match(jd_struct["must_have"])
        imp_score = compute_match(jd_struct["important"])
        nice_score = compute_match(jd_struct["nice_to_have"])
        impl_score = compute_match(jd_struct["implicit"])

        final_score = (
            must_score * weights["must_have"] +
            imp_score * weights["important"] +
            nice_score * weights["nice_to_have"] +
            impl_score * weights["implicit"]
        )

        results.append({
            "candidate": resumes[i]["filename"],
            "must_have": must_score,
            "important": imp_score,
            "nice_to_have": nice_score,
            "implicit": impl_score,
            "final_score": final_score
        })

    return results

In [41]:
weighted_scores = SkillMatcher(metadata_list, jd_struct)

for r in weighted_scores:
    print(r)

{'candidate': 'Resume-RahulSaini.pdf', 'must_have': 0.0, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.25, 'final_score': 0.025}
{'candidate': 'Resume-NehaYadav.pdf', 'must_have': 0.0, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.0, 'final_score': 0.0}
{'candidate': 'Resume-RiyaSharma.pdf', 'must_have': 0.75, 'important': 0.5, 'nice_to_have': 0.5, 'implicit': 0.0, 'final_score': 0.55}
{'candidate': 'Resume-KunalVerma.pdf', 'must_have': 0.25, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.0, 'final_score': 0.1}
{'candidate': 'Resume-SnehaKulKarni.pdf', 'must_have': 0.25, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.25, 'final_score': 0.125}
{'candidate': 'Resume-ArjunMehta.pdf', 'must_have': 0.75, 'important': 0.5, 'nice_to_have': 0.5, 'implicit': 0.0, 'final_score': 0.55}


In [42]:
sorted_scores = sorted(weighted_scores, key=lambda x: x["final_score"], reverse=True)

for r in sorted_scores:
    print(r)

{'candidate': 'Resume-RiyaSharma.pdf', 'must_have': 0.75, 'important': 0.5, 'nice_to_have': 0.5, 'implicit': 0.0, 'final_score': 0.55}
{'candidate': 'Resume-ArjunMehta.pdf', 'must_have': 0.75, 'important': 0.5, 'nice_to_have': 0.5, 'implicit': 0.0, 'final_score': 0.55}
{'candidate': 'Resume-SnehaKulKarni.pdf', 'must_have': 0.25, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.25, 'final_score': 0.125}
{'candidate': 'Resume-KunalVerma.pdf', 'must_have': 0.25, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.0, 'final_score': 0.1}
{'candidate': 'Resume-RahulSaini.pdf', 'must_have': 0.0, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.25, 'final_score': 0.025}
{'candidate': 'Resume-NehaYadav.pdf', 'must_have': 0.0, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.0, 'final_score': 0.0}


Now We Apply a Threshold of selection(currently its 0.3) what we plan is to make it Dynamic as well in future use case

In [43]:
def ApplyThreshold(weighted_scores, threshold=0.3):

    selected = []
    rejected = []

    for r in weighted_scores:
        if r["final_score"] >= threshold:
            selected.append(r)
        else:
            rejected.append(r)

    return selected, rejected

In [44]:
selected, rejected = ApplyThreshold(weighted_scores, threshold=0.3)

print("SELECTED:\n")
for r in selected:
    print(r)

print("\nREJECTED:\n")
for r in rejected:
    print(r)

SELECTED:

{'candidate': 'Resume-RiyaSharma.pdf', 'must_have': 0.75, 'important': 0.5, 'nice_to_have': 0.5, 'implicit': 0.0, 'final_score': 0.55}
{'candidate': 'Resume-ArjunMehta.pdf', 'must_have': 0.75, 'important': 0.5, 'nice_to_have': 0.5, 'implicit': 0.0, 'final_score': 0.55}

REJECTED:

{'candidate': 'Resume-RahulSaini.pdf', 'must_have': 0.0, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.25, 'final_score': 0.025}
{'candidate': 'Resume-NehaYadav.pdf', 'must_have': 0.0, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.0, 'final_score': 0.0}
{'candidate': 'Resume-KunalVerma.pdf', 'must_have': 0.25, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.0, 'final_score': 0.1}
{'candidate': 'Resume-SnehaKulKarni.pdf', 'must_have': 0.25, 'important': 0.0, 'nice_to_have': 0.0, 'implicit': 0.25, 'final_score': 0.125}


Now we have Selected but when multiple and similar, how do we compare right? for that purpose we have another tool WhosBetter()- using LLM Again.

In [45]:
def WhosBetter(candidates, jd_text):

    prompt = f"""
    You are an expert hiring assistant.

    Given the following Job Description:
    {jd_text}

    And the following candidates:

    {candidates}

    Task:
    - Rank the candidates from best to worst
    - Consider depth of experience, relevance to GenAI, and practical skills

    Output format:
    1. Candidate Name - Reason
    2. Candidate Name - Reason
    ...
    """

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

WhyThisPerson()- A summary of Resume and whether the person is good for JD or not? (LLM EDITION) for extra info and comparison.

In [23]:
def WhyThisPerson(resume_text, jd_text):

    prompt = f"""
    Compare the candidate with the job description.

    Job Description:
    {jd_text}

    Candidate Resume:
    {resume_text}

    Give a concise evaluation.

    Output format:

    Strengths:
    - ...

    Gaps:
    - ...
    """

    response = client.chat.completions.create(
        model="stepfun/step-3.5-flash:free",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content

Hero()- Our Agent Dispatcher

In [24]:
def Hero(weighted_scores, resumes, texts, jd_text):

    # sort candidates
    sorted_scores = sorted(weighted_scores, key=lambda x: x["final_score"], reverse=True)

    results = []

    strong = []
    borderline = []
    weak = []

    # classify
    for r in sorted_scores:
        score = r["final_score"]

        if score > 0.5:
            strong.append(r)
        elif score > 0.3:
            borderline.append(r)
        else:
            weak.append(r)

    print("=== STRONG CANDIDATES ===\n")

    # explain strong
    for rank, r in enumerate(strong, start=1):

        idx = [i for i, res in enumerate(resumes) if res["filename"] == r["candidate"]][0]

        explanation = WhyThisPerson(texts[idx], jd_text)

        print(f"Candidate: {r['candidate']}")
        print(f"Rank: {rank}")
        print(f"Score: {r['final_score']:.3f}")
        print("Confidence: High")
        print(explanation)
        print("\n" + "="*50 + "\n")

    # borderline → compare
    if len(borderline) > 1:
        print("=== BORDERLINE COMPARISON ===\n")

        candidate_texts = ""
        for r in borderline:
            idx = [i for i, res in enumerate(resumes) if res["filename"] == r["candidate"]][0]
            candidate_texts += f"\nCandidate: {r['candidate']}\n{texts[idx]}\n"

        comparison = WhosBetter(candidate_texts, jd_text)
        print(comparison)

    # weak → reject
    print("\n=== REJECTED CANDIDATES ===\n")

    for r in weak:
        print(f"{r['candidate']} | Score: {r['final_score']:.3f} | Confidence: Low")

In [25]:
Hero(weighted_scores, resumes, texts, jd_text)

=== STRONG CANDIDATES ===

Candidate: Resume-RiyaSharma.pdf
Rank: 1
Score: 0.550
Confidence: High
**Strengths:**
- Direct experience with core requirements: LangChain (agents), Pinecone (vector DB), and Python.
- Demonstrated deployment expertise with Docker, Kubernetes, and REST APIs (FastAPI).
- Relevant project experience building LLM-based applications (chatbot with tool usage) and using HuggingFace models.
- Covers preferred ML fundamentals via PyTorch and practical application.

**Gaps:**
- No explicit mention of FAISS (only Pinecone listed for vector databases).
- "RAG pipelines" are implied via vector DB implementation but not explicitly stated in experience.


Candidate: Resume-ArjunMehta.pdf
Rank: 2
Score: 0.550
Confidence: High
**Strengths:**
- Directly matches all core requirements: strong Python, hands-on RAG pipeline experience (built RAG assistant, MultiDoc QA), LangChain usage, and FAISS vector database familiarity.
- Demonstrates relevant deployment skills with Docker 

Now UI Time- Since the UI was not Functioning Well with the LLM(Too many Calls Happening in the backend), I have Decided for now to utilize the  UI  with the Precomputed Data Only

In [34]:
TOP_K = 3
selected_sorted = sorted(selected, key=lambda x: x["final_score"], reverse=True)[:TOP_K]

final_outputs = ""

# ✅ create maps ONCE
name_map = {
    resumes[i]["filename"]: candidate_names[i]
    for i in range(len(resumes))
}

index_map = {
    resumes[i]["filename"]: i
    for i in range(len(resumes))
}

for rank, r in enumerate(selected_sorted, start=1):

    # ✅ get correct index
    idx = index_map[r["candidate"]]

    # ✅ get display name
    candidate_name = name_map[r["candidate"]]

    # ✅ pass correct resume text
    explanation = WhyThisPerson(texts[idx], jd_text)

    score = r["final_score"]

    if score > 0.5:
        confidence = "High"
    elif score > 0.3:
        confidence = "Medium"
    else:
        confidence = "Low"

    final_outputs += f"""
📌 Candidate: {candidate_name}
🏆 Rank: {rank}
📊 Score: {score:.3f}
🎯 Confidence: {confidence}

{explanation}
----------------------------------------
"""

In [35]:
import gradio as gr

# --- Wrapper Functions ---

def build_database(zip_file):
    global resumes_loaded

    if zip_file is None:
        return "❌ Please upload a ZIP file."

    # 🔥 Demo mode — no heavy processing
    resumes_loaded = True

    return "✅ EmployeeBase ready (Demo Mode - Precomputed Data Loaded)"


def run_pipeline():
    global final_outputs

    # safety checks
    if 'resumes_loaded' not in globals():
        return "❌ Please upload resumes first."

    try:
        return final_outputs  # 🔥 precomputed results
    except:
        return "❌ Please run backend pipeline first (precompute results)."


# --- UI Layout ---

with gr.Blocks() as demo:

    gr.Markdown("# 🚀 Hire-Er-Key (GenAI Engineer Edition) V1.0")

    gr.Markdown("""
Hire-Er-Key (Pronounced as "hierarchy") is a tool utilized by "The Hiring Hierarchy"
for efficient and intelligent hiring of future GenAI Engineers.

✨ A RAG-based tool built for the Hiring Hierarchy.
""")

    with gr.Group():
        gr.Markdown("## 📂 Upload Resumes (ZIP only)")
        file_input = gr.File(file_types=[".zip"])
        upload_btn = gr.Button("⚙️ Build EmployeeBase")
        upload_status = gr.Textbox(label="Status")

    with gr.Group():
        gr.Markdown("## 🎯 Candidate Evaluation")
        run_btn = gr.Button("🔥 Show Eligible Candidates")
        with gr.Group():
          gr.Markdown("### Output")
          output_box = gr.Markdown()

    upload_btn.click(fn=build_database, inputs=file_input, outputs=upload_status)
    run_btn.click(fn=run_pipeline, outputs=output_box)


demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8ce7b5f946bd41028b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8ce7b5f946bd41028b.gradio.live


### 🚀 Version 2.0 — Planned Enhancements

To improve flexibility, realism, and usability, the following upgrades are proposed:

* **Dynamic Job Description Support**
  → Allow the system to adapt to *any role*, not just GenAI Engineer

* **Dynamic Thresholding**
  → Adjust selection criteria based on role type (internship, entry-level, senior)

* **Top-K Candidate Selection**
  → Return only the most relevant candidates instead of all filtered ones

* **Candidate Name Extraction**
  → Replace file-based identifiers with actual candidate names

* **Semantic Skill Equivalence**
  → Use LLM reasoning to match equivalent skills (e.g., “LLM orchestration” ≈ LangGraph)


🚀 V2 MINIMAL IMPLEMENTATION

Added:

✅ 1. Candidate Name Extraction

✅ 2. Top-K Selection